# Ejercicio: Web Scraping

**Nombre:** Bautista Alexis  
**Fecha:** 08 de julio de 2026

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [4]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [5]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [6]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [7]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

#### 1. Definir una estructura para agrupar los datos

In [9]:
receta_estructurada = {
    "titulo": title,
    "descripcion": description,
    "ingredientes": ingredients,
    "instrucciones": instructions,
    "informacion_nutricional": nutrition_facts
}

# Convertir la estructura en un único bloque de texto optimizado para RAG
documento_rag = f"""
Título de la Receta: {receta_estructurada['titulo']}
Descripción: {receta_estructurada['descripcion']}

Ingredientes:
{chr(10).join(['- ' + ing for ing in receta_estructurada['ingredientes']])}

Instrucciones de Preparación:
{chr(10).join([f'{i+1}. {inst}' for i, inst in enumerate(receta_estructurada['instrucciones'])])}

Información Nutricional:
{chr(10).join(['- ' + info for info in receta_estructurada['informacion_nutricional']])}
"""

# Comprobación del texto generado
print(documento_rag)


Título de la Receta: Rotisserie Chicken
Descripción: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.

Ingredientes:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper

Instrucciones de Preparación:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, st

#### 2. Chunking y Embeddings

In [ ]:
import os
import json
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


os.environ["GOOGLE_API_KEY"] = "Google API"

soup = BeautifulSoup(html_content, "html.parser")

# Extraer el bloque JSON-LD limpio en lugar de parsear etiquetas HTML sueltas
schema_script = soup.find("script", id="allrecipes-schema_1-0", type="application/ld+json")
recipe_data = json.loads(schema_script.string)[0] # El JSON es una lista, tomamos el primer elemento

# Chunking Lógico Semántico
# En lugar de cortar por caracteres, creamos "Chunks" (Documentos) con sentido completo.
documentos_rag = []

# Chunk A: Metadatos, Descripción e Ingredientes
titulo = recipe_data.get("name", "Receta sin título")
ingredientes = "\n".join([f"- {ing}" for ing in recipe_data.get("recipeIngredient", [])])
nutricion = recipe_data.get("nutrition", {})
texto_base = f"Receta: {titulo}\nDescripción: {recipe_data.get('description', '')}\n\nIngredientes:\n{ingredientes}\n\nCalorías: {nutricion.get('calories', 'N/A')}"

documentos_rag.append(Document(
    page_content=texto_base,
    metadata={"source": titulo, "section": "ingredientes_y_resumen"}
))

# Chunk B: Instrucciones de Preparación
instrucciones_raw = recipe_data.get("recipeInstructions", [])
instrucciones = "\n".join([f"{i+1}. {paso.get('text', '')}" for i, paso in enumerate(instrucciones_raw)])

documentos_rag.append(Document(
    page_content=f"Instrucciones para {titulo}:\n{instrucciones}",
    metadata={"source": titulo, "section": "instrucciones"}
))

# Chunk C: Reseñas de los usuarios
# Filtramos y creamos un chunk por cada reseña para capturar el contexto de la experiencia del usuario.
reviews = recipe_data.get("review", [])
for review in reviews:
    autor = review.get("author", {}).get("name", "Anónimo")
    rating = review.get("reviewRating", {}).get("ratingValue", "N/A")
    comentario = review.get("reviewBody", "")
    
    if comentario: # Solo indexar si hay un comentario escrito
        texto_review = f"Reseña de {titulo} por {autor} (Puntuación: {rating}/5):\n{comentario}"
        documentos_rag.append(Document(
            page_content=texto_review,
            metadata={"source": titulo, "section": "review", "author": autor}
        ))

print(f"Se generaron {len(documentos_rag)} chunks lógicos listos para el RAG.")

# Generación de Embeddings e instanciación de ChromaDB
embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

vectorstore = Chroma.from_documents(
    documents=documentos_rag, 
    embedding=embeddings_model,
    collection_name="recetas_jsonld_collection"
)

print("Embeddings semánticos almacenados en ChromaDB")

Se generaron 20 chunks lógicos listos para el RAG.
Embeddings semánticos almacenados en ChromaDB


#### 3. Recuperación (Retriever)

In [30]:
# Convertimos la base de datos vectorial (vectorstore) en un recuperador.
# search_kwargs={"k": 3} le indica al sistema que devuelva los 3 documentos más relevantes.
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} 
)

# Definimos una consulta de prueba.
pregunta_usuario = "Recetas de pollo asado"

# Realizamos la búsqueda en nuestra base de datos vectorial
documentos_recuperados = retriever.invoke(pregunta_usuario)

# Imprimimos los resultados para verificar qué información recuperó el SRI
print(f"Pregunta: '{pregunta_usuario}'\n")
print(f"Se encontraron {len(documentos_recuperados)} fragmentos relevantes:\n")
print("-" * 50)

for i, doc in enumerate(documentos_recuperados, 1):
    print(f"\n--- Documento {i} ---")
    print(f"Sección de origen: {doc.metadata.get('section', 'Desconocida')}")
    if 'author' in doc.metadata:
        print(f"Autor de la reseña: {doc.metadata['author']}")
    print(f"\nContenido extraído:\n{doc.page_content}\n")
    print("-" * 50)

Pregunta: 'Recetas de pollo asado'

Se encontraron 3 fragmentos relevantes:

--------------------------------------------------

--- Documento 1 ---
Sección de origen: ingredientes_y_resumen

Contenido extraído:
Receta: Rotisserie Chicken
Descripción: Rotisserie chicken that&#39;s easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.

Ingredientes:
- 1 (3 pound) whole chicken
- 1 pinch salt
- 0.25 cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- 0.25 tablespoon ground black pepper

Calorías: 357 kcal

--------------------------------------------------

--- Documento 2 ---
Sección de origen: ingredientes_y_resumen

Contenido extraído:
Receta: Rotisserie Chicken
Descripción: Rotisserie chicken that&#39;s easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.

Ingredientes:
- 1 (3 pound) whole chicken
- 1 pinch salt
- 0.25 cup butte

#### 4. Generación

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Instanciar el LLM de Gemini
# temperature=0.2 ayuda a que el modelo sea más preciso y menos creativo
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Formatear los documentos recuperados en el Paso 3
def formatear_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

contexto_formateado = formatear_documentos(documentos_recuperados)

# Diseñar el Prompt Template
# Aquí establecemos las "reglas" para el modelo.
template_rag = """
Eres un asistente culinario experto. Tu tarea es responder a la pregunta del usuario utilizando ÚNICAMENTE el siguiente contexto recuperado de recetas y reseñas.

Si la respuesta no se encuentra en el contexto proporcionado, responde simplemente "No tengo suficiente información en mis recetas para responder a esto". No intentes inventar la respuesta.

Contexto Recuperado:
{context}

Pregunta del Usuario: 
{question}

Respuesta:
"""

prompt = PromptTemplate.from_template(template_rag)

# Construir y ejecutar la cadena de generación (Pipeline)
cadena_generacion = prompt | llm | StrOutputParser()

# Ejecutamos la cadena pasando las variables necesarias.
# Si la API de Gemini no tiene cuota disponible, devolvemos una respuesta de respaldo
# para que la celda no falle
try:
    respuesta_final = cadena_generacion.invoke({
        "context": contexto_formateado,
        "question": pregunta_usuario
    })
except Exception as error:
    respuesta_final = (
        "No se pudo generar una respuesta con Gemini en este entorno. "
        f"Motivo: {error}\n\n"
        "Contexto recuperado de respaldo:\n"
        f"{contexto_formateado[:1000]}"
    )

# Resultados
print("Respuesta generada por el LLM\n")
print(respuesta_final)

Respuesta generada por el LLM

Aquí tienes una receta de pollo asado:

**Receta: Rotisserie Chicken**

**Descripción:** Pollo asado que es fácil de cocinar en una parrilla de gas y resulta húmedo y jugoso con piel crujiente. Esta es una receta sencilla que a nuestra familia le encanta.

**Ingredientes:**
*   1 (3 libras) pollo entero
*   1 pizca de sal
*   0.25 taza de mantequilla, derretida
*   1 cucharada de sal
*   1 cucharada de pimentón molido
*   0.25 cucharada de pimienta negra molida

**Calorías:** 357 kcal

**Instrucciones:**
1.  Reúne todos los ingredientes. Precalienta una parrilla exterior a fuego alto y engrasa ligeramente la rejilla.
2.  Sazona la cavidad del pollo con una pizca de sal. Ata las patas con hilo de cocina; luego ata las alas al ave. Asegura el pollo en un accesorio de asador.
3.  Coloca el asador sobre la parrilla precalentada y cocina durante 10 minutos.
4.  Mientras tanto, mezcla rápidamente la mantequilla, 1 cucharada de sal, pimentón y pimienta. Baja la 

##### Ampliacion del ejemplo

In [34]:
# Ampliamos el número de documentos recuperados para asegurar que capturemos reseñas
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # Aumentamos de 3 a 5
)

# Ejemplo A: Extracción de parámetros directos
pregunta_valida_1 = "¿Cuánta mantequilla y sal se necesita exactamente para preparar el pollo?"

# Ejecutamos la cadena
respuesta_final = cadena_generacion.invoke({
    "context": formatear_documentos(retriever.invoke(pregunta_valida_1)),
    "question": pregunta_valida_1
})

print("Resultado\n", respuesta_final)

Resultado
 Se necesita 0.25 taza de mantequilla derretida. En cuanto a la sal, se usa 1 pizca para sazonar la cavidad del pollo y 1 cucharada para la mezcla de adobo.
